In [7]:
import numpy as np
import matplotlib.pyplot as plt
import re
import json
import pandas as pd
import os
import time
import threading
from http.server import SimpleHTTPRequestHandler
from socketserver import TCPServer
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pickle

cur_path = os.getcwd()

In [2]:
# Load the local Excel file
file_path = 'Drug parameters.xlsx'
# Skip the first two header rows
df = pd.read_excel(file_path, sheet_name='IC50', skiprows=2, header=None)

# Channel mapping for columns C through I
channels = ['INa', 'IKr', 'ICaL', 'INaL', 'IKs', 'Ito', 'IK1']

# Regex for IC50 and Hill coefficient: matches "Value(Hill)"
ic50_pattern = re.compile(r"([0-9.]+)\(([0-9.]+)\)")

# Regex for EFTPCmax: extracts the first numeric sequence (e.g., from "0.155#")
eftp_pattern = re.compile(r"([0-9.]+)")

drug_dict = {}

# Forward fill the EFTPCmax column (index 9) to handle merged cells
df[9] = df[9].ffill()

for index, row in df.iterrows():
    # Column B (index 1) is the Drug Name
    drug_name = str(row[1]).strip()
    
    # Skip empty rows
    if drug_name == 'nan' or not drug_name:
        continue
        
    drug_dict[drug_name] = {}

    # 1. Extract IC50 and Hill coefficient for each channel (Columns C-I)
    for i, channel in enumerate(channels):
        col_idx = i + 2
        cell_value = str(row[col_idx])
        
        if cell_value != 'nan' and cell_value != 'None' and cell_value.strip():
            match = ic50_pattern.search(cell_value)
            if match:
                drug_dict[drug_name][channel] = {
                    "IC50": float(match.group(1)),
                    "h": float(match.group(2))
                }
            else:
                # Fallback for IC50 only
                val_match = eftp_pattern.search(cell_value)
                if val_match:
                    drug_dict[drug_name][channel] = {
                        "IC50": float(val_match.group(1)),
                        "h": None
                    }

    # 2. Extract EFTPCmax (Column J / Index 9)
    eftp_cell = str(row[9])
    if eftp_cell != 'nan' and eftp_cell.strip():
        eftp_match = eftp_pattern.search(eftp_cell)
        if eftp_match:
            drug_dict[drug_name]['EFTPCmax'] = float(eftp_match.group(1))
        else:
            drug_dict[drug_name]['EFTPCmax'] = None
    else:
        drug_dict[drug_name]['EFTPCmax'] = None

# --- Usage Example ---
drug = "Amiodarone I"
if drug in drug_dict:
    data = drug_dict[drug]
    print(f"--- {drug} Parameters ---")
    print(f"EFTPCmax: {data['EFTPCmax']} µM")
    print(f"IKr IC50: {data.get('IKr', {}).get('IC50')} µM")
    print(f"IKr Hill (h): {data.get('IKr', {}).get('h')}")

--- Amiodarone I Parameters ---
EFTPCmax: 0.155 µM
IKr IC50: 0.86 µM
IKr Hill (h): 1.09


In [3]:
# save drug_dict as pkl
with open('drug_dict.pkl', 'wb') as f:
    pickle.dump(drug_dict, f)

In [4]:
def run_simulation_for_drug(drug_name):
    if drug_name not in drug_dict:
        print(f"Drug '{drug_name}' not found in the dataset.")
        return
    
    drug_data = drug_dict[drug_name]
    drug_data['drug_name'] = drug_name

    # 1. --- make other unmentioned currents values ---
    all_currents = ['INa', 'IKr', 'ICaL', 'INaL', 'IKs', 'Ito', 'IK1']
    for current in all_currents:
        if current not in drug_data:
            drug_data[current] = {
                "IC50": 0.0,
                "h": 1.0
            }

    # 2. --- OUTPUT TO JS ---
    output_filename = 'drug_data.js'
    output_path = './2D-TNNP-pacing-general-5xdrug'

    with open(f"{output_path}/{output_filename}", 'w') as js_file:
        js_file.write("const drugData = ")
        json.dump(drug_data, js_file, indent=4)
        js_file.write(";")  # End the JS variable declaration

    # 3. --- put the js data inside html file
    idx_file = './2D-TNNP-pacing-general-5xdrug/index.html'

    # place it after <script src='Abubu/libs/Abubu.js'></script>, if already exist, continue, otherwise add it
    with open(idx_file, 'r') as file:
        html_content = file.read()
        script_tag = f"<script src='{output_filename}'></script>"
        if script_tag not in html_content:
            insertion_point = html_content.find("<script src='Abubu/libs/Abubu.js'></script>") + len("<script src='Abubu/libs/Abubu.js'></script>")
            new_html_content = html_content[:insertion_point] + f"\n<script src='{output_filename}'></script>\n" + html_content[insertion_point:]
            with open(idx_file, 'w') as file:
                file.write(new_html_content)

    # 4 --- run simulation in chrome

    PORT = 8000
    DIRECTORY = "2D-TNNP-pacing-general-5xdrug" # The folder containing your index.html
    TARGET_MESSAGE = "simulation finished"
    URL = f"http://localhost:{PORT}/index.html"

    def start_server():
        """Starts a local server in the specified directory."""
        os.chdir(os.path.abspath(DIRECTORY))
        # Allow restarting the script immediately without "Address already in use" errors
        TCPServer.allow_reuse_address = True
        with TCPServer(("", PORT), SimpleHTTPRequestHandler) as httpd:
            print(f"Serving at {URL}")
            httpd.serve_forever()
        
    # 1. Start the server in a background thread so the script can keep moving
    server_thread = threading.Thread(target=start_server, daemon=True)
    server_thread.start()

    # 2. Configure Chrome
    options = webdriver.ChromeOptions()
    options.set_capability('goog:loggingPrefs', {'browser': 'ALL'})
    # Optional: This keeps the driver logs quiet in your terminal
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    options.add_argument("--window-position=-500,1350")
    driver = webdriver.Chrome(options=options)

    try:
        # 3. Open the localhost URL
        driver.get(URL)
        print("Simulation started on localhost. Monitoring console...")
    
        # wait for 10 s
        time.sleep(10)
        # --- NEW: Automatically click the Solve/Pause button ---
        try:
            # 1. Look for the span containing 'Solve/Pause'
            # We use '*' because dat.GUI doesn't use standard <button> tags
            xpath_selector = "//*[contains(text(), 'Solve/Pause')]"
            
            # 2. Wait for the element to be present and visible
            solve_element = WebDriverWait(driver, 2).until(
                EC.visibility_of_element_located((By.XPATH, xpath_selector))
            )
            
            # 3. Click the element directly via Selenium
            solve_element.click()
            print("Clicked 'Solve/Pause' GUI element successfully.")
        except Exception as e:
            print(f"Could not find or click the button automatically: {e}")
        # -------------------------------------------------------
        running = True
        while running:
            logs = driver.get_log('browser')
            for entry in logs:
                # entry['message'] often contains extra info, so we check if our string is IN it
                if TARGET_MESSAGE.lower() in entry['message'].lower():
                    print(f"Match found: '{TARGET_MESSAGE}'. Finalizing...")
                    time.sleep(5) # Give you a moment to see the final state
                    running = False
                    break
            time.sleep(1)

    finally:
        print("Shutting down...")
        driver.quit()
        # The server thread will die automatically because it's a 'daemon'
    


In [5]:
need_to_redo = ['test1(cisapride)','test2(verapamil)','Amiodarone II',
 'Bepridil II',
 'Bepridil III',
 'Chloropromazine II',
 'Cisapride II',
 'Diltiazem II',
 'Dofetilide II',
 'Dofetilide III',
 'Flecainide II',
 'Flecainide III',
 'Lidocaine II',
 'Mexiletine II',
 'Mibefradil II',
 'Moxifloxacin II',
 'Moxifloxacin III',
 'Nilotinib II',
 'Quinidine',
 'Ranolazine',
 'Saquinavir',
 'Sertindole II',
 'Sotalol II',
 'Sparfloxacin II',
 'Terfenadine II',
 'Verapamil II',
 'Verapamil III']

need_to_redo = ['test3(none)']


In [ ]:
#for drug_name in drug_dict.keys():
indicator = False
for drug_name in drug_dict.keys():
    os.chdir(cur_path)
    print(f"Running simulation for {drug_name}...")
    run_simulation_for_drug(drug_name)

Running simulation for Primidone...
Serving at http://localhost:8000/index.html


127.0.0.1 - - [09/Apr/2026 14:59:59] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 14:59:59] "GET /libs/dat.gui.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 14:59:59] "GET /libs/stats.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 14:59:59] "GET /Abubu/libs/Abubu.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 14:59:59] "GET /drug_data.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 14:59:59] "GET /config.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 14:59:59] "GET /libs/require.js HTTP/1.1" 200 -


Simulation started on localhost. Monitoring console...


127.0.0.1 - - [09/Apr/2026 15:00:00] code 404, message File not found
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /app/main.js?bust=1775761199837 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /libs/shader.js?bust=1775761199837 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /ComputeGL/ComputeGL.js?bust=1775761199837 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /libs/text.js?bust=1775761199837 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /app/shaders/vertShader.vert?bust=1775761199837&bust=1775761199837 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /app/shaders/initShader.frag?bust=1775761199837&bust=1775761199837 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /app/shaders/compShader.frag?bust=1775761199837&bust=1775761199837 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:00:00] "GET /app/shaders/getCurrentsShader.frag?bust=1775761199837&bust=1775761

Clicked 'Solve/Pause' GUI element successfully.
Match found: 'simulation finished'. Finalizing...
Shutting down...
Running simulation for Procainamide...
Serving at http://localhost:8000/index.html


...

Clicked 'Solve/Pause' GUI element successfully.
Match found: 'simulation finished'. Finalizing...
Shutting down...
Running simulation for test3(none)...
Serving at http://localhost:8000/index.html


127.0.0.1 - - [09/Apr/2026 15:17:20] "GET /index.html HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:20] "GET /libs/dat.gui.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /libs/stats.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /Abubu/libs/Abubu.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /drug_data.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /config.js HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /libs/require.js HTTP/1.1" 200 -


Simulation started on localhost. Monitoring console...


127.0.0.1 - - [09/Apr/2026 15:17:21] code 404, message File not found
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /app/main.js?bust=1775762241207 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /libs/shader.js?bust=1775762241207 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /ComputeGL/ComputeGL.js?bust=1775762241207 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:21] "GET /libs/text.js?bust=1775762241207 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:22] "GET /app/shaders/vertShader.vert?bust=1775762241207&bust=1775762241207 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:22] "GET /app/shaders/initShader.frag?bust=1775762241207&bust=1775762241207 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:22] "GET /app/shaders/compShader.frag?bust=1775762241207&bust=1775762241207 HTTP/1.1" 200 -
127.0.0.1 - - [09/Apr/2026 15:17:22] "GET /app/shaders/getCurrentsShader.frag?bust=1775762241207&bust=1775762

Clicked 'Solve/Pause' GUI element successfully.
Match found: 'simulation finished'. Finalizing...
Shutting down...
